# Spectral analysis and inference with jnwb

Synthetic data only, so this runs anywhere `pip install jnwb` works. Two channels share a 20 Hz rhythm during a 'response' window; the notebook finds it in time-frequency power, measures coherence, and tests the response against baseline with a cluster permutation test.

In [ ]:
import numpy as np
import jnwb

rng = np.random.default_rng(0)
fs = 1000.0
n_trials, n_times = 40, 1500
t = np.arange(n_times) / fs
window = (t >= 0.6) & (t < 1.1)            # response window, 0.6-1.1 s
shared = np.sin(2 * np.pi * 20.0 * t) * window
ch1 = shared + rng.normal(scale=1.0, size=(n_trials, n_times))
ch2 = shared + rng.normal(scale=1.0, size=(n_trials, n_times))
ch1.shape

## Time-frequency power

`complex_tfr` returns complex coefficients plus `coi_mask`, which marks samples too close to the edges for the wavelet at that frequency. Mask them before averaging.

In [ ]:
freqs = np.arange(8.0, 41.0, 2.0)
tfr = jnwb.complex_tfr(ch1, fs=fs, freqs=freqs)      # (trials, freqs, times)
power = np.where(tfr.coi_mask, tfr.power, np.nan)
mean_power = np.nanmean(power, axis=0)                # (freqs, times)
peak_freq = freqs[np.nanargmax(np.nanmean(mean_power[:, window], axis=1))]
print(f'peak frequency in the response window: {peak_freq:.0f} Hz')
assert peak_freq == 20.0

## Decibels last

Average power first, divide by baseline, take `10*log10` once. `aggregate_to_db` enforces that order; averaging decibels would bias each trial by its own noise level.

In [ ]:
beta = (freqs >= 14) & (freqs <= 30)
base = (t >= 0.2) & (t < 0.5)
resp_power = np.nanmean(power[:, beta][:, :, window], axis=(1, 2))   # per trial
base_power = np.nanmean(power[:, beta][:, :, base], axis=(1, 2))
db = jnwb.aggregate_to_db(resp_power, base_power, how='mean_of_ratios', aggregate_over=0)
print(f'beta response re baseline: {float(db):.1f} dB')
assert float(db) > 0

## Coherence between the channels

The band definitions decide every band p-value, so `cross_area_coherence` requires them. The null is a circular shift of one channel; `p_value_floor` is the smallest p-value the chosen number of surrogates can produce.

In [ ]:
coh = jnwb.cross_area_coherence(
    ch1[:, window].ravel(), ch2[:, window].ravel(), fs=fs,
    freq_bands={'beta': (14.0, 30.0), 'gamma': (30.0, 80.0)},
    n_surrogates=99, rng=np.random.default_rng(1),
)
for band in ('beta', 'gamma'):
    print(f"{band:5s} coherence {coh['band_coherence'][band]:.3f}  p = {coh['band_significance'][band]:.3f}")
print('p-value floor:', round(coh['p_value_floor'], 4))
assert coh['band_coherence']['beta'] > coh['band_coherence']['gamma']

## Response versus baseline, over time

`cluster_permutation_test` compares two sets of trials sample by sample and corrects across time by cluster mass, so one test covers the whole window.

In [ ]:
beta_env = np.nanmean(power[:, beta], axis=1)             # (trials, times)
baseline_env = np.repeat(np.nanmean(beta_env[:, base], axis=1, keepdims=True), n_times, axis=1)
keep = np.all(np.isfinite(beta_env), axis=0)
out = jnwb.cluster_permutation_test(
    beta_env[:, keep], baseline_env[:, keep], n_permutations=199, rng=np.random.default_rng(2)
)
best = min(out['clusters'], key=lambda c: c['p_value'])
print('smallest cluster p-value:', best['p_value'])
assert best['p_value'] < 0.05